<a href="https://colab.research.google.com/github/bcbutler2212/Education-Policy-Research-Chatbot/blob/Sofia/working_llama_3_2_1b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.2-1B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.2-1B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [2]:
from huggingface_hub import login
login(new_session=False)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline
import os
from google.colab import userdata

# Set the HF_TOKEN environment variable
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")

In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

In [ ]:
# load files temporarily --> have to do this manually at this point but shouldnt in the future.
from google.colab import files
uploaded = files.upload()

In [ ]:
# unzip
!unzip /content/pdfs_for_chatbot.zip

In [ ]:
!pip install langchain[all] langchain-community chromadb sentence-transformers transformers pypdf
# note * I had trouble with different langchain versions so I am using the classic

In [ ]:
!pip install langchain-classic

In [10]:
# imports below
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.vectorstores import Chroma
from langchain_classic.embeddings import SentenceTransformerEmbeddings
from langchain_classic.llms import HuggingFacePipeline
from langchain_classic.document_loaders import PyPDFLoader
from transformers import LlamaTokenizer, LlamaForCausalLM, pipeline



In [11]:
from langchain_classic.chains import RetrievalQA

In [12]:

from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
import os
os.listdir("/content")

In [ ]:
# Load all pdfs (815 right now)

pdf_folder = "/content/pdfs_for_chatbot"

docs = []
for file in os.listdir(pdf_folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder, file))
        loaded_docs = loader.load()

        # Add source data for citations
        for i, doc in enumerate(loaded_docs):
            doc.metadata = {
                "source": f"{file} page {i+1}"
            }
        docs.extend(loader.load())

print(f"Loaded {len(docs)} PDF pages")

In [ ]:
# Split long text into smaller chunks --> this creates 3765 chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = splitter.split_documents(docs)

print(f"Created {len(split_docs)} text chunks")

In [ ]:
# wrap
#from langchain_community.llms import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

In [17]:
#embedding model
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_classic.vectorstores import Chroma

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [19]:
# Create a persistent Chroma DB
vectordb = Chroma.from_documents(split_docs, embedding=embedding_model, persist_directory="chroma_db")


In [ ]:
vectordb.persist()

In [ ]:
# If chunks print then working
results = vectordb.similarity_search("What is this document about?", k=2)
for i, r in enumerate(results, 1):
    print(f"\nResult {i}:\n{r.page_content[:500]}...")

In [22]:
# make retriever --> return 3 ? change this?
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [24]:
from langchain_classic.chains import RetrievalQA

# connect llama with chroma
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

In [ ]:
# query the chatbot
question = "What is education policy?"
result = qa_chain.invoke({"query": question})
print(result["result"])

In [ ]:
#this prints the sources
if "source_documents" in result:
    sources = set()
    for doc in result["source_documents"]:
        src = doc.metadata.get("source", "unknown")
        sources.add(src)

    print("\nSources:")
    for src in sources:
        print("-", src)

In [31]:
# Colab form UI for easy use
from IPython.display import display
import ipywidgets as widgets


In [35]:
# create textbox to collect question
input_box = widgets.Text(
    description='Ask:',
    placeholder='Type your question...',
    layout=widgets.Layout(width='70%')
)
output_box = widgets.Output()

button = widgets.Button( # submit button
    description='Submit',
    button_style='primary'
)

def on_submit(_): # query question when user clicks submit
    output_box.clear_output()
    query = input_box.value

    with output_box:
        if not query.strip(): # prompt question if blank
            print("Please enter a question.")
            return

        # Run the QA chain
        result = qa_chain.invoke({"query": query})
        answer = result["result"]

        # Extract sources - only filenames at this point but want more specific references for the future
        sources = "\n".join(
            f"- {doc.metadata.get('source', 'unknown')}"
            for doc in result["source_documents"]
        )

        print(f"Answer:\n{answer}\n")
        print("Sources:")
        print(sources)

button.on_click(on_submit)


In [ ]:
display(input_box, button, output_box)
# ask about promise scholarships --> should cite from Gurantz's work

In [ ]:
'''
Sofia's TO DO:
- need to add similarity threshold to prevent LLM from answering with no context or make new query function
- Find pages from citation / link to direct lines within the documents
'''